# Lab 6: Daily Planner / To-Do Agent

**AI Agent Lab 6 of 8**

---

## What this lab does

In this lab you will build a Daily Planner Agent. You give it commands in plain English - things like 'add buy milk', 'show tasks', 'done 2', 'remove 3', or 'suggest' - and it manages a to-do list stored as a list of dictionaries. Each task has a title, a done/not-done status, and a priority level. The 'suggest' command returns an AI-style (hardcoded) prioritisation recommendation. The whole demo runs on a hardcoded list of commands.

## What you will learn

**Python concepts:**
- How to manage a list of dictionaries as a simple in-memory database
- How to add, update, and remove items from a list
- How to use string methods like `.startswith()` and `.split()` to parse commands
- How to sort a list of dictionaries by a key value
- How to format and print a table from a list of dictionaries

**Agent concepts:**
- Command parsing as a form of intent detection
- State management: the task list is the agent's persistent state
- AI-style suggestions based on heuristic rules

## Important things to know before starting

- **What is state?** State means the data the agent keeps in memory between commands. Here, the task list is the state. Every command reads or modifies it.
- **What is command parsing?** Parsing a command means breaking it into a verb and its arguments. For example, 'add buy milk' -> verb: 'add', argument: 'buy milk'.
- **What is a heuristic?** A heuristic is a rule of thumb used to make a decision without a full calculation. Our 'suggest' command uses heuristics to recommend which task to do next.
- **This lab runs completely offline.**

## How the agent works

```
Command string (e.g., 'add buy milk')
        |
        v
  Parse: split into command verb and argument
        |
        v
  Act: add / show / mark done / remove / suggest
  (all operations on the task list dictionary)
        |
        v
  Report: print the updated task list or suggestion
```

---
## Step 0: Libraries and Setup

This lab uses **only built-in Python libraries**.

We use:
- `datetime` - to record when each task was added
- `sys` - to print the Python version

In [ ]:
# No installation needed
# Optional: # !pip install nothing

import sys
print('Setup check passed!')
print(f'Python version: {sys.version}')

---
## Importing Libraries

| Library | What it is used for in this lab |
|---|---|
| `datetime` | Records the date and time when each task is created |

In [ ]:
import datetime  # used to timestamp tasks when they are added

print('Libraries imported successfully.')

---
## Step 1: Set Up the Task List and Priority System

**What this block does:** We define the main task list (`list_of_all_tasks`) as an empty list. We also define the priority levels the agent uses when auto-assigning priority.

**Why we need it:** The task list is the agent's state - it holds all tasks. The priority system lets the agent intelligently auto-assign priorities based on task keywords.

**What to expect as output:** Confirmation that the task list is ready.

In [ ]:
# The main to-do list. Each item is a dictionary with these keys:
# 'id' (int), 'title' (str), 'done' (bool), 'priority' (str), 'added_at' (str)
# This list is the agent's persistent state - all commands read or modify it.
list_of_all_tasks = []

# A counter for assigning unique IDs to tasks
# This variable is used in add_task() to give each task a unique number
next_task_id_counter = 1

# Priority levels - used when auto-detecting task urgency
high_priority_trigger_words = ['urgent', 'important', 'critical', 'asap', 'deadline', 'today', 'emergency']
low_priority_trigger_words = ['maybe', 'someday', 'eventually', 'optional', 'if time']

print('Task list initialised (empty).')
print(f'High-priority trigger words: {high_priority_trigger_words}')
print(f'Low-priority trigger words: {low_priority_trigger_words}')

---
## Step 2: Build the Core Task Operations

**What this block does:** We define four helper functions:
- `add_task(title)` - creates a new task dictionary and appends it to the list
- `show_all_tasks()` - prints all tasks in a formatted table
- `mark_task_done(task_id)` - finds a task by ID and sets its `done` flag to True
- `remove_task(task_id)` - removes a task from the list by its ID

**Why we need it:** These functions are the building blocks the agent calls when it processes each command.

**What to expect as output:** Functions are defined. Confirmation is printed.

In [ ]:
def determine_task_priority(task_title):
    """
    Automatically assigns a priority level to a task based on its title.

    Takes in:
      task_title - the title of the task (string)

    Returns:
      'High', 'Low', or 'Medium' (string)
    """
    lowered_title = task_title.lower()
    for trigger_word in high_priority_trigger_words:
        if trigger_word in lowered_title:
            return 'High'
    for trigger_word in low_priority_trigger_words:
        if trigger_word in lowered_title:
            return 'Low'
    return 'Medium'


def add_task(task_title):
    """
    Adds a new task to list_of_all_tasks.

    Takes in:
      task_title - the name of the task to add (string)

    Returns:
      A confirmation message string.
    """
    global next_task_id_counter  # we need to modify the counter variable

    assigned_priority = determine_task_priority(task_title)
    new_task_entry = {
        'id': next_task_id_counter,
        'title': task_title,
        'done': False,
        'priority': assigned_priority,
        'added_at': datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    }
    list_of_all_tasks.append(new_task_entry)
    next_task_id_counter = next_task_id_counter + 1
    return f'Task added: [{assigned_priority}] {task_title} (ID: {new_task_entry["id"]})'


print('Task operation functions being defined...')

In [ ]:
def show_all_tasks():
    """
    Prints all tasks in the list as a formatted table.

    Takes in: nothing
    Returns: nothing - prints directly to screen.
    """
    if not list_of_all_tasks:
        print('  Your task list is empty! Add some tasks with "add <task name>".')
        return

    # Sort tasks: High priority first, then Medium, then Low; done tasks at the bottom
    priority_sort_values = {'High': 0, 'Medium': 1, 'Low': 2}
    sorted_task_list = sorted(
        list_of_all_tasks,
        key=lambda task: (task['done'], priority_sort_values.get(task['priority'], 1))
    )

    print(f'  {"ID":<5} {"Done":<6} {"Priority":<10} {"Task"}')
    print('  ' + '-' * 55)
    for task_item in sorted_task_list:
        done_marker = 'Yes' if task_item['done'] else 'No'
        print(f"  {task_item['id']:<5} {done_marker:<6} {task_item['priority']:<10} {task_item['title']}")
    print(f'  Total: {len(list_of_all_tasks)} task(s)')


def mark_task_done(task_id_to_mark):
    """
    Marks the task with the given ID as done.

    Takes in:
      task_id_to_mark - the integer ID of the task to mark done

    Returns:
      A confirmation or error message (string).
    """
    for task_item in list_of_all_tasks:
        if task_item['id'] == task_id_to_mark:
            task_item['done'] = True
            return f"Task {task_id_to_mark} marked as done: '{task_item['title']}'"
    return f'Error: no task found with ID {task_id_to_mark}.'


def remove_task(task_id_to_remove):
    """
    Removes the task with the given ID from the list.

    Takes in:
      task_id_to_remove - the integer ID of the task to remove

    Returns:
      A confirmation or error message (string).
    """
    for position, task_item in enumerate(list_of_all_tasks):
        if task_item['id'] == task_id_to_remove:
            removed_task = list_of_all_tasks.pop(position)
            return f"Task {task_id_to_remove} removed: '{removed_task['title']}'"
    return f'Error: no task found with ID {task_id_to_remove}.'


print('All task operation functions defined.')

---
## Step 3: Build the AI Suggestion Function

**What this block does:** We define `suggest_next_task()`. It looks at all incomplete tasks and recommends the one to do next, based on priority (High before Medium before Low). It prints a hardcoded AI-style recommendation message.

**Why we need it:** The 'suggest' command gives the agent a taste of AI-powered task management. Real AI task planners use more complex scheduling, but the idea is the same.

**What to expect as output:** The function is defined.

In [ ]:
# SIMULATED AI: In a real system this would use an LLM to analyse tasks.
# Here, we use priority rules to make a recommendation.

def suggest_next_task():
    """
    Recommends the next task to work on based on priority.

    Looks at incomplete tasks only.
    Recommends the highest-priority incomplete task.

    Takes in: nothing
    Returns: a recommendation string.
    """
    incomplete_tasks = [task for task in list_of_all_tasks if not task['done']]

    if not incomplete_tasks:
        return 'All tasks are complete! Great job - you are all caught up.'

    priority_order = ['High', 'Medium', 'Low']
    for priority_level in priority_order:
        tasks_at_this_priority = [task for task in incomplete_tasks if task['priority'] == priority_level]
        if tasks_at_this_priority:
            top_task = tasks_at_this_priority[0]
            return (
                f'AI Suggestion: Focus on task {top_task["id"]} first.\n'
                f'  Title: "{top_task["title"]}"\n'
                f'  Priority: {top_task["priority"]}\n'
                f'  (There are {len(incomplete_tasks)} incomplete tasks in total.)'
            )

    return 'Could not determine a suggestion.'


print('suggest_next_task() defined.')

---
## Step 4: Build the Command Parser and Main Agent

**What this block does:** We define `parse_command()` to split a raw command string into a verb and its arguments. Then we define `daily_planner_agent()`, the main agent that takes a command, parses it, and calls the right function.

**Why we need it:** Separating parsing from action makes the code cleaner. The parser is like the agent's 'ears' - it turns raw text into structured input.

**What to expect as output:** Both functions are defined.

In [ ]:
def parse_command(raw_command_string):
    """
    Splits a command string into a verb and the rest of the text.

    Examples:
      'add buy milk'    -> ('add', 'buy milk')
      'done 3'          -> ('done', '3')
      'show tasks'      -> ('show', 'tasks')
      'suggest'         -> ('suggest', '')

    Takes in:
      raw_command_string - the command the user typed (string)

    Returns:
      A tuple: (command_verb_string, argument_string)
    """
    stripped_command = raw_command_string.strip().lower()
    parts = stripped_command.split(' ', 1)  # split into at most 2 parts
    command_verb = parts[0]
    command_argument = parts[1] if len(parts) > 1 else ''
    return (command_verb, command_argument)


def daily_planner_agent(raw_command_string):
    """
    The main Daily Planner Agent.

    Parses a command and calls the appropriate task operation.

    Supported commands:
      add <task title>  - adds a new task
      show              - displays all tasks
      done <id>         - marks task <id> as done
      remove <id>       - removes task <id>
      suggest           - recommends what to do next

    Takes in:
      raw_command_string - the user's command (string)

    Returns:
      Nothing - prints the result to screen.
    """

    command_verb, command_argument = parse_command(raw_command_string)

    print(f'\n> Command: "{raw_command_string}"')

    if command_verb == 'add':
        if not command_argument:
            print('  Error: please provide a task title. Example: add buy milk')
        else:
            result_message = add_task(command_argument)
            print(f'  {result_message}')

    elif command_verb in ('show', 'list', 'tasks'):
        print('  Current task list:')
        show_all_tasks()

    elif command_verb == 'done':
        if not command_argument.isdigit():
            print('  Error: please provide a valid task ID number. Example: done 2')
        else:
            result_message = mark_task_done(int(command_argument))
            print(f'  {result_message}')

    elif command_verb == 'remove':
        if not command_argument.isdigit():
            print('  Error: please provide a valid task ID number. Example: remove 3')
        else:
            result_message = remove_task(int(command_argument))
            print(f'  {result_message}')

    elif command_verb == 'suggest':
        suggestion_text = suggest_next_task()
        print(f'  {suggestion_text}')

    else:
        print(f'  Unknown command: "{command_verb}". Try: add, show, done, remove, suggest')


print('daily_planner_agent() defined. Ready to accept commands.')

---
## Step 5: Run the Agent on Sample Commands

**What this block does:** We define a list of 15 sample commands and run the agent on each one. The commands cover adding tasks, showing the list, marking tasks done, removing tasks, getting suggestions, and some edge cases.

**Why we need it:** This is the main demo. You will see the task list grow and change as commands are processed, just like a real to-do app.

**What to expect as output:** A printed log of every command and the agent's response.

In [ ]:
# Reset the task list before running the demo
list_of_all_tasks.clear()
next_task_id_counter = 1

# These commands simulate a user interacting with the planner throughout their day
list_of_sample_commands = [
    'add submit assignment by deadline today',
    'add buy groceries after work',
    'add call mom',
    'add URGENT: fix the bug in the login page',
    'add maybe read a book someday',
    'add prepare presentation for Friday meeting',
    'show',
    'suggest',
    'done 1',
    'done 4',
    'show',
    'suggest',
    'remove 5',
    'add water the plants if time allows',
    'show',
    'done 99',          # edge case: task ID does not exist
    'remove abc',       # edge case: non-numeric ID
    'unknown command',  # edge case: invalid command verb
]

print('=' * 60)
print('DAILY PLANNER AGENT - DEMO RUN')
print('=' * 60)

for single_command in list_of_sample_commands:
    daily_planner_agent(single_command)

---
## Summary

You built a **Daily Planner / To-Do Agent** that manages a task list with commands, auto-priority detection, and AI-style suggestions.

### Key takeaways
- **Lists of dictionaries as a mini-database:** This pattern is the basis of many real data management systems.
- **Command parsing is intent detection:** Splitting a string into verb + argument is the same idea as detecting intents in a chatbot.
- **State management:** The task list persists across commands because it is a global variable. Agents must manage their state carefully.
- **Auto-priority with keyword rules:** Automatically assigning priority based on keywords in the task title is a practical AI-lite feature.
- **Edge case resilience:** The agent handles invalid IDs, missing arguments, and unknown commands without crashing.

### Connection to the next lab

In **Lab 7**, you will build a Meeting Summarizer Agent. Given a meeting transcript, it will extract attendees, action items, and decisions, then produce a formatted summary and a follow-up email.

---

## Try It Yourself

1. **Add a deadline field:** When adding a task, check if the command contains 'by <date>' and store the deadline in the task dictionary. Display it in the `show_all_tasks()` table.

2. **Add a 'clear done' command:** Add a new command `'clear done'` that removes all tasks where `done` is True. Test it by adding tasks, marking some done, then running `clear done`.

3. **Save to a file:** After the demo runs, write code that saves `list_of_all_tasks` to a text file using `json.dumps()`. Then write code that reads it back. This simulates persistence between sessions.